# Book RAG Project

Your task is to build a **retrieval-augmented generation (RAG)** system that answers questions about **one book**, and shows the passages of the book that each answer is based on. Every group works on a different book from Project Gutenberg; the table in the README shows which book is yours.

![The project in two parts. Part 1, once per book: 1. get your book, 2. clean the text, 3. split it into chunks, 4. embed the chunks and store them in a Chroma vector store. Part 2, for each question: 5. retrieve the passages from the vector store, 6. generate the answer, and return the answer with its passages](assets/book_rag_project_workflow.png)

# The Business Problem

A public library wants to offer a **reading companion** on its website: readers ask a question about a book in their own words, for example "Why does the Tin Woodman want a heart?", and receive a short answer. Readers must be able to check every answer, so the companion also shows **the passages of the book that the answer is based on**. When the book does not contain the answer, the companion has to say so instead of making one up.

A large language model alone already knows a lot about famous books, but it does not cite passages, and it can confuse details or invent them, especially for less well-known books. Your RAG system retrieves the passages from the book first and lets the model answer **only from these passages**, so that every answer comes with the passages a reader can check.

Before the library builds the website, its staff want to **try the companion themselves**. Your last task is therefore a **command-line prototype**: a small script that librarians run in the terminal to ask questions about your book and read the answers with their passages.

# Data Description

The data ships with this repository in the folder `data/`: **eight books** from [Project Gutenberg](https://www.gutenberg.org/), one plain text file per book (UTF-8). Your group works with **one** of them.

| File | Title | Author | First published | Kind | Words |
| :-- | :-- | :-- | :-- | :-- | --: |
| `adventures_of_pinocchio_collodi.txt` | The Adventures of Pinocchio | Carlo Collodi | 1883 | children's novel, translated from Italian | 39,900 |
| `adventures_of_sherlock_holmes_doyle.txt` | The Adventures of Sherlock Holmes | Arthur Conan Doyle | 1892 | twelve detective stories | 104,500 |
| `antarctic_penguins_levick.txt` | Antarctic Penguins: A Study of Their Social Habits | G. Murray Levick | 1914 | natural history | 30,600 |
| `climate_in_relation_to_man_ward.txt` | Climate, Considered Especially in Relation to Man | Robert DeCourcy Ward | 1918 | climatology textbook (second edition) | 88,300 |
| `frankenstein_shelley.txt` | Frankenstein; or, The Modern Prometheus | Mary Wollstonecraft Shelley | 1818 | gothic novel | 75,000 |
| `the_time_machine_wells.txt` | The Time Machine | H. G. Wells | 1895 | science fiction novella | 32,500 |
| `tom_browns_school_days_hughes.txt` | Tom Brown's School Days | Thomas Hughes | 1857 | boarding school novel | 105,400 |
| `wonderful_wizard_of_oz_baum.txt` | The Wonderful Wizard of Oz | L. Frank Baum | 1900 | children's fantasy novel | 39,600 |

The word counts leave out the licence text. Every file is exactly as Project Gutenberg distributes it:

- It **starts with a header** (title, author, release date) and the line `*** START OF THE PROJECT GUTENBERG EBOOK ... ***`, and it **ends with** the line `*** END OF THE PROJECT GUTENBERG EBOOK ... ***`, followed by the Project Gutenberg licence. Only the text between these two lines belongs to the book.
- The lines are **wrapped after about 70 characters**, so a single sentence is spread over several lines. A blank line separates two paragraphs.
- Some books contain notes of the transcribers, a table of contents, and placeholders such as `[Illustration]`.

The books are in the public domain in the United States; the Project Gutenberg licence at the end of each file explains how they may be used.

## Task

1. **Get your book.** Set `BOOK` to the file name of your book and read it with `Path.read_text`. Print the first and last 1,500 characters: what comes before and after the book itself?
2. **Clean the text.** Keep only the text between the `*** START OF ...` and `*** END OF ...` lines, and join the wrapped lines of each paragraph with `re.sub`. Store the result in a [`Document`](https://reference.langchain.com/python/langchain-core/documents/base/Document) with the title and author in its `metadata`.
3. **Split into chunks.** Use [`RecursiveCharacterTextSplitter`](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter) with a `chunk_size` and `chunk_overlap` of your choice, and `add_start_index=True`, so that every chunk stores its position in the book. How many chunks do you get?
4. **Embed and store.** Embed the chunks with [`HuggingFaceEmbeddings`](https://reference.langchain.com/python/langchain-huggingface/embeddings/huggingface/HuggingFaceEmbeddings) (`sentence-transformers/all-MiniLM-L6-v2`) and store them in a [`Chroma`](https://reference.langchain.com/python/langchain-chroma/vectorstores/Chroma) vector store with `persist_directory="chroma_db"`.

> [!IMPORTANT]
> Pass your own `ids` to `add_documents`, for example `f"{BOOK}-{i}"`. Without them, every run of the cell adds all chunks again.

5. **Retrieve.** Create a retriever with `as_retriever(search_kwargs={"k": 4})` and `invoke` it with a few questions about your book. Print the chunks with their `start_index`: do they contain the answer?
6. **Generate.** Build an LCEL chain `prompt | llm | parser` from a [`ChatPromptTemplate`](https://reference.langchain.com/python/langchain-core/prompts/chat/ChatPromptTemplate) that tells the model to answer **only from the passages**, [`ChatGroq`](https://reference.langchain.com/python/langchain-groq/chat_models/ChatGroq) with `openai/gpt-oss-20b` and `temperature=0`, and a `StrOutputParser`. Write `ask(question)`, which returns the answer and the `start_index` of each passage. Try your questions from step 5, and one whose answer is not in the book.
7. **Build the command-line prototype.** Copy your code into `rag_cli.py` and complete the `while` loop described in its comments, so that it reads questions with `input()` until the user types `quit`, and prints each answer and its passage in colour. Run it with `uv run python rag_cli.py`.

> [!NOTE]
> The free Groq plan limits requests per minute. If you get a rate limit error, wait a minute and try again.

In [1]:
# Import the libraries you need for this project
import hashlib
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()  # reads GROQ_API_KEY from the .env file

True

# 1. Get the Book

In [2]:
# Step 1: get the book
BOOK = "antarctic_penguins_levick.txt"
raw_text = Path("data", BOOK).read_text(encoding="utf-8")

print(f"{len(raw_text):,} characters")

195,227 characters


In [3]:
print("=== FIRST 1,500 CHARACTERS ===")
print(raw_text[:1500])

=== FIRST 1,500 CHARACTERS ===
The Project Gutenberg eBook of Antarctic Penguins: A Study of Their Social Habits
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: Antarctic Penguins: A Study of Their Social Habits

Author: G. Murray Levick


        
Release date: August 1, 2011 [eBook #36922]

Language: English

Other information and formats: www.gutenberg.org/ebooks/36922

Credits: Produced by Jana Srna and the Online Distributed
        Proofreading Team at http://www.pgdp.net (This file was
        produced from images generously made available by
        Biodiversity Heritage Librar

In [4]:
print("\n=== LAST 1,500 CHARACTERS ===")
print(raw_text[-1500:])


=== LAST 1,500 CHARACTERS ===
licited donations from donors in such states who
approach us with offers to donate.

International donations are gratefully accepted, but we cannot make
any statements concerning tax treatment of donations received from
outside the United States. U.S. laws alone swamp our small staff.

Please check the Project Gutenberg web pages for current donation
methods and addresses. Donations are accepted in a number of other
ways including checks, online payments and credit card donations. To
donate, please visit: www.gutenberg.org/donate.

Section 5. General Information About Project Gutenberg electronic works

Professor Michael S. Hart was the originator of the Project
Gutenberg concept of a library of electronic works that could be
freely shared with anyone. For forty years, he produced and
distributed Project Gutenberg eBooks with only a loose network of
volunteer support.

Project Gutenberg eBooks are often created from several printed
editions, all of which 

# 2. Clean the Text

In [5]:
# Step 2: clean the text
# 2a) keep only what is between the START and END lines
start = re.search(r"\*\*\* START OF.*?\*\*\*", raw_text)
end = re.search(r"\*\*\* END OF.*?\*\*\*", raw_text)
body = raw_text[start.end() : end.start()]
print(f"Length of the book inbetween START and END lines: {len(body):,}.")

# 2b) drop front matter (ads, title page, contents, list of illustrations):
#     the real text starts at the LAST "INTRODUCTION" heading line (the first is in the contents)
intro = list(re.finditer(r"^\s*INTRODUCTION\s*$", body, flags=re.M))[-1]
body = body[intro.start() :]
print(f"Length of the book after dropping front matter: {len(body):,}.")

# 2c) drop the printer's line and the transcriber's list of corrections at the end
body = body[: body.rindex("PRINTED AT")]
print(f"Length of the book after dropping end matter: {len(body):,}.")

# 2d) remove placeholders like [Illustration: ...] (they can span several lines)
body = re.sub(r"\[Illustration[^\]]*\]", "", body)
print(f"Length of the book after removing illustrations: {len(body):,}.")

# 2e) join wrapped lines: a single newline becomes a space, blank lines (paragraph breaks) stay
body = re.sub(r"(?<!\n)[ \t]*\n[ \t]*(?!\n)", " ", body)
body = re.sub(r"[ \t]+", " ", body)  # collapse repeated spaces
body = re.sub(r"\n\s*\n+", "\n\n", body).strip()  # one blank line between paragraphs
print(f"Length of the book after joining wrapped lines: {len(body):,}.\n")

doc = Document(
    page_content=body,
    metadata={"title": "Antarctic Penguins: A Study of Their Social Habits", "author": "G. Murray Levick"},
)
print(f"{len(doc.page_content):,} characters, {len(doc.page_content.split()):,} words\n")
print("Excerpt:\n")
print(doc.page_content[:800])
print("...")
print(doc.page_content[-600:])

Length of the book inbetween START and END lines: 175,675.
Length of the book after dropping front matter: 168,095.
Length of the book after dropping end matter: 167,109.
Length of the book after removing illustrations: 161,478.
Length of the book after joining wrapped lines: 158,705.

158,705 characters, 28,738 words

Excerpt:

INTRODUCTION

The penguins of the Antarctic regions very rightly have been termed the true inhabitants of that country. The species is of great antiquity, fossil remains of their ancestors having been found, which showed that they flourished as far back as the eocene epoch. To a degree far in advance of any other bird, the penguin has adapted itself to the sea as a means of livelihood, so that it rivals the very fishes. This proficiency in the water has been gained at the expense of its power of flight, but this is a matter of small moment, as it happens.

 (1) _Pygoscelis adeliae._

In few other regions could such an animal as the penguin rear its young, for w

# 3. Split the Text into Chunks

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,  # every chunk stores its position in the book in metadata["start_index"]
)
chunks = splitter.split_documents([doc])

print(f"{len(chunks)} chunks")
print(f"average length: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} characters")
print(chunks[5].metadata)
print(chunks[5].page_content)

231 chunks
average length: 720 characters
{'title': 'Antarctic Penguins: A Study of Their Social Habits', 'author': 'G. Murray Levick', 'start_index': 3892}
To continue the sketch, I quote two other writers:

M. Racovitza, of the "Belgica" expedition, well describes them as follows:

"Imagine a little man, standing erect, provided with two broad paddles instead of arms, with head small in comparison with the plump stout body; imagine this creature with his back covered with a black coat ... tapering behind to a pointed tail that drags on the ground, and adorned in front with a glossy white breast-plate. Have this creature walk on his two feet, and give him at the same time a droll little waddle, and a pert movement of the head; you have before you something irresistibly attractive and comical."

Dr. Louis Gain, of the French Antarctic expedition, gives us the following description:


# 4. Embed Chunks and Store in Vector DataBase  

In [7]:
# The embedding model turns each chunk into a vector of 384 numbers
# (the first run downloads the model, about 90 MB)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector = embeddings.embed_query("How do penguins build their nests?")
print(f"vector with {len(vector)} numbers, starting with {[round(x, 3) for x in vector[:5]]}")

/Users/albert/projects/amle/gutenberg-book-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1106 11:09:30.066000 52335 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 35309.63it/s]


vector with 384 numbers, starting with [0.004, 0.044, 0.002, 0.053, 0.001]


In [8]:
vector_store = Chroma(
    collection_name="penguins",
    embedding_function=embeddings,
    persist_directory="chroma_db",
)

# Idempotent: start from an empty collection, so running this cell again (or after changing
# the chunking) always leaves exactly the current chunks in the store.
vector_store.reset_collection()

# Content-based ids: the SHA-1 hash of the chunk text
ids = [hashlib.sha1(chunk.page_content.encode("utf-8")).hexdigest() for chunk in chunks]
vector_store.add_documents(chunks, ids=ids)

print(f"{vector_store._collection.count()} chunks in the vector store")

231 chunks in the vector store


# 5. Create the Retriever

In [9]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

questions = [
    "How do Adelie penguins build their nests?",
    "Why do the cocks fight?",
    "What do skua gulls do to the penguins?",
    "What is the capital of France?",  # not in the book
]

for question in questions:
    print(f"\n=== {question} ===")
    for passage in retriever.invoke(question):
        print(f"\n[start_index {passage.metadata['start_index']}]")
        print(passage.page_content)


=== How do Adelie penguins build their nests? ===

[start_index 22929]
On October 18 the weather cleared and a fair number of penguins started to build their nests. The great majority however, apparently resting, still sat about. Those that built took their stones from old nests, as at present so many of these lay unoccupied. They made quite large nests, some inches high at the sides, with a comfortable hollow in the middle to sit in. The stone carrying (Fig. 20) was done by the male birds, the hens keeping continual guard over the nest, as otherwise the pair would have been robbed of the fruits of their labours as fast as they were acquired.

[start_index 6446]
The Adelies, as will be seen in these pages, are extremely brave, and though panic occasionally overtakes them, I have seen a bird return time after time to attack a seaman who was brutally sending it flying by kicks from his sea-boot, before I arrived to interfere. An exact description of the plumage of the Adelie penguins wi

# 6. Generate 

7. **Build the command-line prototype.** Copy your code into `rag_cli.py` and complete the `while` loop described in its comments, so that it reads questions with `input()` until the user types `quit`, and prints each answer and its passage in colour. Run it with `uv run python rag_cli.py`.

> [!NOTE]
> The free Groq plan limits requests per minute. If you get a rate limit error, wait a minute and try again.

In [10]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a reading companion for the book 'Antarctic Penguins' by G. Murray Levick. "
            "Answer the question using ONLY the passages below. "
            'If the passages do not contain the answer, say exactly: "The book does not answer this question." '
            "Do not use any other knowledge. Keep the answer short.\n\n"
            "Passages:\n{context}",
        ),
        ("human", "{question}"),
    ]
)

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# LCEL chain: fill the prompt -> call the model -> turn the reply into a plain string
chain = prompt | llm | StrOutputParser()

In [11]:
def ask(question):
    """Answer a question from the book. Returns the answer and the start_index of each passage used."""
    passages = retriever.invoke(question)
    context = "\n\n".join(f"[Passage {i}]\n{p.page_content}" for i, p in enumerate(passages, 1))
    answer = chain.invoke({"context": context, "question": question})
    return answer, [p.metadata["start_index"] for p in passages]


for question in questions:  # the questions from step 5, including the one that is not in the book
    answer, start_indices = ask(question)
    print(f"\n=== {question} ===")
    print(answer)
    print(f"passages at start_index: {start_indices}")


=== How do Adelie penguins build their nests? ===
Adelie penguins build their nests by gathering stones from abandoned nests. The males carry the stones while the hens guard the site. They construct large, stone‑walled nests a few inches high on the sides, leaving a comfortable hollow in the middle for sitting.
passages at start_index: [22929, 6446, 67711, 7805]

=== Why do the cocks fight? ===
The book does not answer this question.
passages at start_index: [46372, 38318, 52511, 44223]

=== What do skua gulls do to the penguins? ===
Skua gulls prey on penguin eggs and chicks, pecking out their eyes and then devouring their kidneys. They do not attack adult penguins.
passages at start_index: [147611, 62105, 140849, 63209]

=== What is the capital of France? ===
The book does not answer this question.
passages at start_index: [16353, 9790, 14770, 157030]
